# 01 - Dataset Audit
Notebook ini fokus pada audit dataset mentah dan validasi aturan domain.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    return cwd

PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data import discover_dataset_files, load_tabular_file, normalize_column_names, audit_dataset_file

DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_REPORTS_DIR = PROJECT_ROOT / 'outputs' / 'reports'
OUTPUT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

dataset_files = discover_dataset_files(DATA_RAW_DIR)
print(f'Dataset files found: {len(dataset_files)}')
for idx, fp in enumerate(dataset_files[:10], start=1):
    print(f'{idx}. {fp.name}')

Dataset files found: 1
1. DSL-StrongPasswordData.csv


In [2]:
def print_domain_distribution(frame: pd.DataFrame) -> dict:
    summary = {}
    for col in ['subject', 'sessionindex', 'rep']:
        if col in frame.columns:
            vc = frame[col].value_counts(dropna=False)
            summary[col] = {
                'unique': int(frame[col].nunique(dropna=False)),
                'top_counts': {str(k): int(v) for k, v in vc.head(10).to_dict().items()},
            }
            print(f'- {col}: unique={summary[col]["unique"]}, top_counts={summary[col]["top_counts"]}')
        else:
            summary[col] = {'missing': True}
            print(f'- {col}: column not found')
    return summary

def check_h_ud_dd_relation(frame: pd.DataFrame, tolerance: float = 1e-6) -> dict:
    dd_cols = [c for c in frame.columns if c.startswith('dd.')]
    results = []
    for dd_col in dd_cols:
        parts = dd_col.split('.')
        if len(parts) != 3:
            continue
        key1, key2 = parts[1], parts[2]
        h_col = f'h.{key1}'
        ud_col = f'ud.{key1}.{key2}'
        if h_col not in frame.columns or ud_col not in frame.columns:
            continue
        temp = frame[[h_col, ud_col, dd_col]].dropna()
        if temp.empty:
            continue
        residual = (temp[h_col] + temp[ud_col] - temp[dd_col]).abs()
        row = {
            'dd_col': dd_col,
            'pairs': int(len(temp)),
            'mean_abs': float(residual.mean()),
            'max_abs': float(residual.max()),
            'within_tol_ratio': float((residual <= tolerance).mean()),
        }
        results.append(row)
        print(f"- {dd_col}: pairs={row['pairs']}, mean_abs={row['mean_abs']:.6f}, max_abs={row['max_abs']:.6f}, within_tol={row['within_tol_ratio']:.2%}")
    return {'checked_triplets': len(results), 'triplets': results}

if not dataset_files:
    raise FileNotFoundError('No raw dataset found. Put CSV in data/raw first.')

audit = audit_dataset_file(dataset_files[0])
frame = normalize_column_names(load_tabular_file(dataset_files[0]))
print(f'Loaded: {dataset_files[0].name} | shape={frame.shape}')
print('\nDistribution checks:')
distribution_summary = print_domain_distribution(frame)
print('\nH+UD~DD checks:')
relation_summary = check_h_ud_dd_relation(frame)

summary_path = OUTPUT_REPORTS_DIR / 'audit_summary.json'
payload = {
    'file': str(dataset_files[0]),
    'shape': audit.shape,
    'columns': audit.columns,
    'distribution': distribution_summary,
    'relation_check': relation_summary,
}
summary_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(f'\nSaved audit summary to: {summary_path}')

Loaded: DSL-StrongPasswordData.csv | shape=(20400, 34)

Distribution checks:
- subject: unique=51, top_counts={'s002': 400, 's003': 400, 's004': 400, 's005': 400, 's007': 400, 's008': 400, 's010': 400, 's011': 400, 's012': 400, 's013': 400}
- sessionindex: unique=8, top_counts={'1': 2550, '2': 2550, '3': 2550, '4': 2550, '5': 2550, '6': 2550, '7': 2550, '8': 2550}
- rep: unique=50, top_counts={'1': 408, '2': 408, '3': 408, '4': 408, '5': 408, '6': 408, '7': 408, '8': 408, '9': 408, '10': 408}

H+UD~DD checks:
- dd.period.t: pairs=20400, mean_abs=0.000000, max_abs=0.000000, within_tol=100.00%
- dd.t.i: pairs=20400, mean_abs=0.000000, max_abs=0.000000, within_tol=100.00%
- dd.i.e: pairs=20400, mean_abs=0.000000, max_abs=0.000000, within_tol=100.00%
- dd.e.five: pairs=20400, mean_abs=0.000000, max_abs=0.000000, within_tol=100.00%
- dd.o.a: pairs=20400, mean_abs=0.000000, max_abs=0.000000, within_tol=100.00%
- dd.a.n: pairs=20400, mean_abs=0.000000, max_abs=0.000000, within_tol=100.00%
- d